In [ ]:
import os, glob
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient

load_dotenv()
w = WorkspaceClient()

CATALOG, SCHEMA = "workspace", "ai_data_team"

# Free Edition provides a serverless SQL warehouse; use the first one
wh = next(iter(w.warehouses.list()))
print("warehouse:", wh.name)

def run(sql):
    r = w.statement_execution.execute_statement(
        statement=sql, warehouse_id=wh.id, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(r.status.error)
    return r

for f in sorted(glob.glob("../data/raw/*.csv")):
    name = os.path.splitext(os.path.basename(f))[0]
    run(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.{name} AS
        SELECT * FROM read_files(
            '/Volumes/{CATALOG}/{SCHEMA}/raw/{name}.csv',
            format => 'csv', header => true, inferSchema => true)
    """)
    n = run(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.{name}").result.data_array[0][0]
    print(f"{name}: {n} rows")

In [ ]:
for t in ["customers", "orders", "monthly_revenue", "product_summary"]:
    r = run(f"SELECT * FROM {CATALOG}.{SCHEMA}.{t} LIMIT 3")
    cols = [c.name for c in r.manifest.schema.columns]
    print(f"\n{t}: {cols}")
    for row in r.result.data_array:
        print("  ", row)